In [0]:
storage_account = "stgecommercedev"
storage_key = "**REDACTED**"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/olist"
gold_path   = f"abfss://gold@{storage_account}.dfs.core.windows.net/olist"
print(f"{gold_path}")

abfss://gold@stgecommercedev.dfs.core.windows.net/olist


In [0]:
from pyspark.sql.functions import (
col, sum as _sum, count, month, year,
date_format, round as _round)

In [0]:
orders=spark.read.format("delta").load(f"{silver_path}/orders")
items=spark.read.format("delta").load(f"{silver_path}/order_items")
reviews=spark.read.format("delta").load(f"{silver_path}/order_reviews")
customers=spark.read.format("delta").load(f"{silver_path}/customers")
products=spark.read.format("delta").load(f"{silver_path}/products")
sellers=spark.read.format("delta").load(f"{silver_path}/sellers")
payments=spark.read.format("delta").load(f"{silver_path}/order_payments")
geolocation=spark.read.format("delta").load(f"{silver_path}/geolocation")
product_category=spark.read.format("delta").load(f"{silver_path}/product_category")

### Revenue by category per month

In [0]:
revenue_by_category = orders \
    .join(items, "order_id") \
    .join(products, "product_id") \
    .join(product_category,"product_category_name")\
    .filter(col("order_status") == "delivered") \
    .withColumn("year_month",
        date_format("order_purchase_timestamp", "yyyy-MM")) \
    .groupBy("year_month", "product_category_name_english") \
    .agg(
        _round(_sum("price"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders")
    ) \
    .orderBy("year_month", "total_revenue", ascending=[True, False])

In [0]:
revenue_by_category.limit(5).display()
print(f"Total number of records:{revenue_by_category.count()}")

year_month,product_category_name_english,total_revenue,total_orders
2016-09,health_beauty,134.97,3
2016-10,furniture_decor,5690.52,65
2016-10,perfumery,4829.1,29
2016-10,toys,4106.5,24
2016-10,consoles_games,3619.36,7


Total number of records:1243


In [0]:
revenue_by_category.write.format("delta")\
    .mode("overwrite")\
    .save(f"{gold_path}/revenue_by_category/")

### Order count by customer state

In [0]:
orders_by_state=orders\
    .join(customers,"customer_id")\
    .join(items,"order_id")\
    .filter(col("order_status") == "delivered")\
    .groupBy("customer_state")\
    .agg(count("order_id").alias("total_revenue"),_round(_sum("price"), 2).alias("total_orders"))\
    .orderBy("total_orders", ascending=False)

orders_by_state.limit(5).display()
print(f"Total number of records:{orders_by_state.count()}")
orders_by_state.write.format("delta")\
    .mode("overwrite")\
    .save(f"{gold_path}/orders_by_state/")

customer_state,total_revenue,total_orders
SP,46448,5067633.16
RJ,14143,1759651.13
MG,12916,1552481.83
RS,6134,728897.47
PR,5649,666063.51


Total number of records:27


### Table conversion

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_gold")

spark.read.format("delta") \
.load(f"{gold_path}/revenue_by_category/") \
.write.format("delta") \
.mode("overwrite") \
.saveAsTable("ecommerce_gold.revenue_by_category")

spark.read.format("delta") \
.load(f"{gold_path}/orders_by_state/") \
.write.format("delta") \
.mode("overwrite") \
.saveAsTable("ecommerce_gold.orders_by_state")